[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/syaikhipin/kdd26-memdiag/blob/tutorial-rebuild/tutorial/phase2_public_datasets/03_benchmarking.ipynb)

> **Run this notebook in Google Colab** — click the badge above. The setup cell auto-clones the repo.


# Phase 2C - Benchmarking (Exercise 2)

**Phase 2C - Exercise 2: benchmarking memory systems.** Independent notebook - runs standalone in Colab or locally (offline, no key).

## 0. Setup (self-contained)

In [ ]:
# Self-contained setup - works standalone in Google Colab or locally.
import sys, os, subprocess
from pathlib import Path
REPO_URL = "https://github.com/syaikhipin/kdd26-memdiag"
try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False
if IN_COLAB:
    repo = Path("/content/kdd26-memdiag")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "tutorial-rebuild", REPO_URL, str(repo)], check=False)
    SOURCE = repo / "source"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy", "matplotlib", "pyyaml"], check=False)
else:
    SOURCE = None
    for cand in [Path.cwd(), *Path.cwd().parents]:
        for sub in ("source", "experiment"):
            if (cand / sub / "run.py").exists():
                SOURCE = cand / sub
                break
        if SOURCE:
            break
    if SOURCE is None:
        raise FileNotFoundError("Run from the repo root (or in Colab it auto-clones).")
sys.path.insert(0, str(SOURCE))
os.environ.setdefault("OPENAI_BASE_URL", "https://api.openai.com/v1")
SOURCE_DIR = SOURCE
PROJECT_ROOT = SOURCE.parent
RESULTS_DIR = PROJECT_ROOT / "results"
print("SOURCE_DIR =", SOURCE, "| IN_COLAB =", IN_COLAB)

print("SOURCE_DIR =", SOURCE_DIR)

## 📖 Narrative: Benchmarking across providers

Exercise 2 runs all four providers on MemoryArena and compares them across:
- **accuracy** (evidence hit rate)
- **latency** (milliseconds per query)
- **cost** (API units consumed)
- **memory size** (storage footprint)

This is the standardized benchmarking pipeline from the proposal: same data, same metrics,
different providers. The key question: **does any single provider dominate?**

## Exercise 2 - benchmark providers on MemoryArena

In [ ]:
import subprocess
cmd = [sys.executable,'-m','benchmark_cli','compare',
 '--providers','verbatim,extracted_facts,episodic,hybrid','--sample_size','20','--benchmark','memoryarena']
r = subprocess.run(cmd, cwd=str(SOURCE_DIR), capture_output=True, text=True)
print(r.stdout[-2200:])

## 📖 Narrative: The benchmarking insight

### 📝 Quick Quiz
1. **Which provider has the highest hit rate on MemoryArena?** Is it the same as LoCoMo?
2. **Which provider uses the most memory?** Why does extracted_facts use more than verbatim?
3. **Look at latency.** Which is fastest? Does memory size correlate with latency?
4. **The key finding:** does any provider dominate across ALL metrics? What does this mean
   for someone choosing a memory system for their own application?

### 💡 Discussion prompt
If you were building a customer-support agent, which provider would you choose?
Consider: the agent needs fast responses (latency), accurate recall (hit rate), and must
fit in 8 GB RAM (memory size). There's no single right answer — it depends on your constraints.

### Discussion
- Does any architecture dominate across **all** datasets? (Expect: no.)
- These dataset-dependent tradeoffs motivate Phase 3.